# 🐻🐂 Bull-Bear Debate Stock Analysis

Run a multi-round **LangGraph** debate between a **Bear** and a **Bull** researcher, judged by an impartial **Judge**, for any company.

This notebook installs dependencies, uploads the source as a ZIP, configures your API key & provider, and runs a debate end-to-end. Tools are mocked in V1; the debate/agent orchestration is the focus.


In [ ]:
#@title 1. Install dependencies
%%capture
!pip install -q langgraph langchain-openai langchain-core tenacity pydantic fastapi uvicorn nest-asyncio
print("Dependencies installed.")

In [ ]:
#@title 2. Upload the source code (ZIP)
# ---------------------------------------------------------------------------
# STEP 1 (on your local machine): build the zip, e.g.
#     cd /path/to/bear_bull_debate
#     python tools/build_bear_bull_debate_zip.py
#   (or: zip -r bear_bull_debate_src.zip src/bear_bull_debate)
#   Zipping the whole project folder also works — the notebook auto-locates
#   the package inside the archive.
#
# STEP 2: run this cell and choose the .zip file when the upload dialog appears.
# ---------------------------------------------------------------------------
from google.colab import files
import zipfile, os, sys, glob

EXTRACT_DIR = "/content/bear_bull_debate_src"
os.makedirs(EXTRACT_DIR, exist_ok=True)

uploaded = files.upload()  # choose your .zip

zipped = [name for name in uploaded if name.endswith(".zip")]
if not zipped:
    raise SystemExit("No .zip file was uploaded.")

for name in zipped:
    with zipfile.ZipFile(name, "r") as z:
        z.extractall(EXTRACT_DIR)
    print(f"Extracted {name}")

# Locate the `bear_bull_debate` package dir (handles nesting from zipping the whole project).
matches = glob.glob(os.path.join(EXTRACT_DIR, "**", "bear_bull_debate"), recursive=True)
if not matches:
    raise SystemExit("Could not find the 'bear_bull_debate' package in the uploaded zip.")

src_root = os.path.dirname(matches[0])
if src_root not in sys.path:
    sys.path.insert(0, src_root)

import bear_bull_debate
print("Source ready:", bear_bull_debate.__file__)

In [ ]:
#@title 3. Configure API key & provider
import os

# ── 1) Choose your provider ─────────────────────────────────────────────
# This automatically sets OPENAI_BASE_URL and the model names below.
PROVIDER = "DeepSeek"  # @param ["OpenAI", "DeepSeek", "Qwen DashScope", "Custom"]

# ── 2) Load your API key (Colab Secrets, or prompt) ─────────────────────
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("Loaded API key from Colab Secrets.")
except Exception:
    import getpass
    os.environ["OPENAI_API_KEY"] = getpass.getpass("API key: ")
    print("API key set from prompt.")

# ── 3) Apply provider defaults (skipped when PROVIDER == "Custom") ───────
PROVIDER_DEFAULTS = {
    "OpenAI": {
        "base_url": None,  # keep OpenAI's default endpoint
        "models": {},
    },
    "DeepSeek": {
        "base_url": "https://api.deepseek.com",
        "models": {
            "BEAR_MODEL": "deepseek-chat",
            "BULL_MODEL": "deepseek-chat",
            "JUDGE_MODEL": "deepseek-chat",
            "SUMMARY_MODEL": "deepseek-chat",
        },
    },
    "Qwen DashScope": {
        "base_url": "https://dashscope.aliyuncs.com/compatible-mode/v1",
        "models": {
            "BEAR_MODEL": "qwen-plus",
            "BULL_MODEL": "qwen-plus",
            "JUDGE_MODEL": "qwen-plus",
            "SUMMARY_MODEL": "qwen-plus",
        },
    },
    "Custom": {"base_url": None, "models": {}},
}
cfg = PROVIDER_DEFAULTS[PROVIDER]
if cfg["base_url"] is not None:
    os.environ["OPENAI_BASE_URL"] = cfg["base_url"]
for key, value in cfg["models"].items():
    os.environ[key] = value

# ── 4) Show the effective config ────────────────────────────────────────
print("\nEffective config:")
print("  PROVIDER        =", PROVIDER)
print("  OPENAI_BASE_URL =", os.environ.get("OPENAI_BASE_URL", "(OpenAI default: api.openai.com)"))
for key in ("BEAR_MODEL", "BULL_MODEL", "JUDGE_MODEL", "SUMMARY_MODEL"):
    print(f"  {key:16s} =", os.environ.get(key))
key = os.environ.get("OPENAI_API_KEY", "")
print("  OPENAI_API_KEY  =", (key[:6] + "..." + key[-4:]) if len(key) > 10 else "(not set)")

# ── 5) Sanity check ─────────────────────────────────────────────────────
if PROVIDER == "Custom" and not os.environ.get("OPENAI_BASE_URL"):
    print("\n⚠️  Custom provider but OPENAI_BASE_URL is not set — requests go to OpenAI.")

In [ ]:
#@title 4. Run a debate
import nest_asyncio
nest_asyncio.apply()

from bear_bull_debate.runner import run_debate

COMPANY = "AAPL"      # @param {type:"string"}
MAX_ROUNDS = 2        # @param {type:"integer"}

result = run_debate(COMPANY, max_rounds=MAX_ROUNDS)

print("Thread ID:", result["thread_id"])
print("\n" + "=" * 70)
print(result["final_report"])
print("=" * 70)
print("\nTool calls performed:")
for log in result["tool_logs"]:
    print("  -", log)

## Customize

- **Provider** — pick it in cell 3 (OpenAI / DeepSeek / Qwen DashScope / Custom); it sets `OPENAI_BASE_URL` and the model names for you.
- **Models** — override `BEAR_MODEL`, `BULL_MODEL`, `JUDGE_MODEL`, `SUMMARY_MODEL` after cell 3 if needed.
- **Rounds** — change `MAX_ROUNDS` (1–5).
- **Async** — use `await run_debate_async(...)` instead (IPython supports top-level await, so `nest_asyncio` is optional).
- **Got a 401?** — your API key didn't match the selected provider. Re-run cell 3 and pick the provider your key belongs to (e.g. DeepSeek), or use a matching key.
